# MCP 複数Tool実践

このNotebookでは、MCP Serverに複数のToolを登録し、OpenAIのLLMが質問内容に応じて適切なToolを選択・実行する流れを確認する。

単にコードを動かすだけではなく、次の点を説明できることを目標とする。

1. MCP Server / Client の役割
2. Python関数をMCP Toolとして公開する仕組み
3. `async` / `await` / `async with` の意味
4. MCP Tool情報をLLMへ渡す流れ
5. LLMがToolを選択し、Clientが実行する流れ
6. Tool追加時にClient側の分岐を増やさなくてよい理由
7. Notebookで検証した処理を `client.py` として単独実行する流れ


## 1. 環境構築

今回の実装環境は以下の通り。

```text
OS            : Windows
Python        : 3.12
conda環境名   : mcp
MCP           : mcp 2.x
LLM API       : OpenAI API
開発環境      : Jupyter Notebook
```

MCP・OpenAI関連ライブラリをインストールし、Jupyterから `mcp` 環境を利用できるようKernelを登録する。

OpenAI APIを利用するため、環境変数 `OPENAI_API_KEY` も設定しておく。

このNotebookでは環境構築そのものよりも、MCP Server / Client / Tool連携の理解を中心に扱う。

### 実行ファイルのディレクトリ構成

```text
C:\python\MCP
├─ server.py
├─ client.py
├─ 04_MCP_複数Tool実践.ipynb
└─ data
    ├─ AI利用ルール.txt
    ├─ PC申請.txt
    ├─ セキュリティ.txt
    ├─ 有給休暇.txt
    └─ 障害対応.txt
```

`server.py` と `client.py` は同じプロジェクトフォルダに配置する。

`client.py` では、

```python
from server import mcp
```

として、同じフォルダ内の `server.py` に定義したMCP Serverを読み込む。


### MCPで扱える主な機能

MCPではToolだけでなく、ServerからClientへ複数種類の機能を提供できる。

#### 今回中心に利用する機能

```text
Tools
→ Python関数などの処理をClientから呼び出す
```

今回の実装では `Tools` を中心に扱う。

Client側では主に、

```text
list_tools()
→ Serverが公開しているTool一覧を取得

call_tool()
→ 指定したToolを実行
```

を使用する。

#### その他の代表的な機能

```text
Resources
→ ファイルやデータなどの情報を提供する

Prompts
→ 再利用可能なプロンプトテンプレートを提供する
```

Client側では、例えば以下のような操作がある。

```text
list_resources()
read_resource()

list_prompts()
get_prompt()
```

#### 発展的な機能

```text
Sampling
→ Server側からLLMによる生成を要求する仕組み

Roots
→ Client側がServerへ利用可能なファイル領域などを示す仕組み

Elicitation
→ Server側からユーザーへ追加情報の入力を求める仕組み
```

今回のポートフォリオでは、MCPの基本的なTool連携を理解することを目的としているため、主に `Tools`、`list_tools()`、`call_tool()` を利用する。


## 2. 実装前に理解しておく `async` / `await` / `async with`

MCP Clientでは非同期処理を利用するため、ServerやClientのコードを見る前にこの3つを確認しておく。

### `async`

`async def` は非同期関数を定義するときに使用する。

```python
async def ask_mcp(question: str):
```

この場合、`ask_mcp()` は通常の関数ではなく、`await` して結果を受け取る非同期関数となる。

---

### `await`

`await` は、**非同期処理の完了を待ち、その結果を受け取る**ために使用する。

```python
tools = await client.list_tools()
```

この例では、`client.list_tools()` の処理が完了するまで待ち、その結果を `tools` に受け取る。

通常の関数と比較すると分かりやすい。

```python
def add(a, b):
    return a + b

result = add(1, 2)
```

通常の関数では、関数側が `return` で値を返し、呼び出し側がその値を受け取る。

非同期関数でも、値そのものは `return` で返す。

```python
async def get_data():
    return "result"
```

ただし、呼び出し側では、

```python
data = await get_data()
```

のように `await` を使い、非同期処理の完了を待って `return` された値を受け取る。

```text
return
→ 関数側が結果を返す

await
→ 呼び出し側が非同期処理の完了を待ち、その結果を受け取る

yield
→ ジェネレータから値を順番に返す
```

したがって、`await` は「非同期版のreturn」ではなく、**非同期関数が返す結果を呼び出し側で待って受け取るための記述**である。

---

### `async with`

`async with` は、非同期オブジェクトの開始・終了処理をまとめて管理する。

```python
async with Client(mcp) as client:
    tools = await client.list_tools()
```

今回の場合、MCP Clientを利用する範囲を `async with` の中にまとめている。

`async with` に入るとClientの利用が開始され、ブロックを抜けると終了処理まで管理される。

---

### Notebookと `.py` ファイルの違い

Jupyter Notebookでは既にイベントループが動いているため、セル内でトップレベルの `await` を利用できる。

一方、通常の `.py` ファイルでは、最後に、

```python
asyncio.run(main())
```

として非同期処理を開始する。


## 3. `server.py` の役割

`server.py` は、MCP Serverを作成し、Clientから利用できるToolを定義する側である。

今回の実装では、

```python
Client(mcp)
```

として、`server.py` で作成したMCP Serverオブジェクトを**同じPythonプロセス内**から直接利用している。

```text
今回の構成

同一PC・同一Pythonプロセス

client.py / Notebook
        ↓
     MCP Client
        ↓
MCP Serverオブジェクト
        ↓
       Tool
        ↓
ローカルファイル
```

この構成は、MCPの仕組みを学習・検証するうえでシンプルで扱いやすい。

ただし、MCP ServerはClientと同じPC・同じプロセスに置く必要はない。

用途によって、例えば次のような構成を取ることができる。

```text
① 同一プロセス
Client(mcp)
→ 今回の構成。学習・テスト・組み込み用途など

② ローカルの別プロセス
Client
↓ stdio
MCP Server
→ ローカルアプリ、CLI、デスクトップアプリなど

③ ネットワーク越し
Client
↓ Streamable HTTP
MCP Server
→ クラウド、社内サーバー、オンプレ環境など
```

つまり、MCPにおけるServerとは「別PCにある物理サーバー」という意味ではなく、**ClientへTool・Resource・Promptなどの機能を提供する役割**を表している。

今回のNotebookでは仕組みの理解を優先して同一プロセス構成を使用し、実運用では用途に応じてローカル別プロセスやリモートServerへ発展させることができる。


### 3-1. MCP Serverを作成する

`server.py` では、まずMCP Serverを作成する。

```python
mcp = MCPServer("My MCP Server")
```

ここでは、

```text
mcp
→ Python側でServerオブジェクトを扱うための変数名

"My MCP Server"
→ MCP Serverに付ける識別用の名前
```

となる。

**どちらも固定名ではなく、用途に応じて変更できる。**

例えば、

```python
company_mcp = MCPServer("Company Document Server")
```

のように定義してもよい。

ただし、変数名を `mcp` から `company_mcp` に変更した場合は、Client側も、

```python
from server import company_mcp
```

のように合わせて変更する必要がある。

また、`server.py` というファイル名自体もMCPで固定されている名前ではない。

例えば、

```text
mcp_server.py
company_server.py
```

などに変更することもできるが、その場合はClient側のimportも、

```python
from mcp_server import mcp
```

のように変更する必要がある。

今回のNotebookでは、役割が分かりやすいため、

```text
server.py
mcp
"My MCP Server"
```

というシンプルな名前を使用している。


### 3-2. Python関数をMCP Toolとして登録する

MCP Toolの本体は、まず通常のPython関数である。

例えば、ファイルの内容を読み込む処理は次のように書ける。

```python
def read_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()
```

このPython関数に `@mcp.tool()` を付けることで、MCP Serverから公開するToolとして登録する。

```python
@mcp.tool()
def read_text_file(path: str) -> str:
    """指定したテキストファイルの内容を読み込みます。"""

    with open(path, "r", encoding="utf-8") as f:
        return f.read()
```

それぞれの役割は次の通り。

```text
@mcp.tool()
→ Python関数をMCP Toolとして登録

def read_text_file(...)
→ 実際に行うPython処理

path: str
→ 引数名と型。Toolのinput_schema生成にも利用される

-> str
→ 戻り値の型

docstring
→ Toolのdescriptionとして利用される
```

このように、MCP用の特殊な処理を一から書くのではなく、**Pythonで作った関数をToolとして公開する**構成になっている。


### 3-3. docstringの重要性

MCP Toolでは、関数のdocstringがToolの `description` として利用される。

例えば、

```python
@mcp.tool()
def count_keyword(
    folder_path: str,
    keyword: str
) -> dict[str, int]:
    """指定したフォルダ内のテキストファイルごとに、キーワードの出現回数を数えます。"""
```

と定義すると、Clientが `list_tools()` でTool情報を取得した際に、このdocstringが `description` として取得される。

LLMは主に、

```text
Tool名
description
input_schema
```

を参考に、「どのToolを使うか」「どの引数を渡すか」を判断する。

そのためdocstringは、単なる人間向けのコード説明ではなく、**LLMがToolの用途を判断するための情報**としても重要になる。

特に似たToolが複数ある場合は、

```text
search_folder_text
→ キーワードを含む行そのものを検索する

count_keyword
→ キーワードが何回出現するかを数える
```

のように、違いが分かる説明を書くことが重要である。


### 3-4. 今回Serverに登録するTool

今回の実践では、次の5種類のToolを利用する。

```text
read_text_file
→ 指定したテキストファイルの内容を取得

list_files
→ フォルダ内のファイル一覧を取得

get_file_info
→ ファイルの基本情報を取得

search_folder_text
→ 複数テキストファイルからキーワードを含む行を検索

count_keyword
→ 各テキストファイル内のキーワード出現回数を取得
```

`count_keyword` も、まずPythonで書いた関数である。

そのPython関数へ `@mcp.tool()` を付けてMCP Toolとして登録し、後からServerへ追加した。

このToolは、**Server側に新しいPython関数を追加してTool登録したとき、Client側に個別の分岐処理を追加しなくても利用できるか**を確認するためにも使用する。


### 3-5. 今回の `server.py`

```python
from pathlib import Path

from mcp.server import MCPServer


mcp = MCPServer("My MCP Server")


@mcp.tool()
def read_text_file(path: str) -> str:
    """指定したテキストファイルの内容を読み込みます。"""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


@mcp.tool()
def list_files(path: str) -> list[str]:
    """指定したフォルダ内のファイル一覧を取得します。"""
    return [p.name for p in Path(path).iterdir()]


@mcp.tool()
def get_file_info(path: str) -> dict:
    """指定したファイルの基本情報を取得します。"""
    file_path = Path(path)

    if not file_path.exists():
        return {
            "name": file_path.name,
            "exists": False
        }

    return {
        "name": file_path.name,
        "suffix": file_path.suffix,
        "size": file_path.stat().st_size,
        "exists": True
    }


@mcp.tool()
def search_folder_text(
    folder_path: str,
    keyword: str
) -> dict[str, list[str]]:
    """指定したフォルダ内のテキストファイルから、キーワードを含む行を検索します。"""

    results = {}

    for path in Path(folder_path).glob("*.txt"):
        with open(path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        matched = [
            line.strip()
            for line in lines
            if keyword in line
        ]

        if matched:
            results[path.name] = matched

    return results


@mcp.tool()
def count_keyword(
    folder_path: str,
    keyword: str
) -> dict[str, int]:
    """指定したフォルダ内のテキストファイルごとに、キーワードの出現回数を数えます。"""

    results = {}

    for path in Path(folder_path).glob("*.txt"):
        text = path.read_text(encoding="utf-8")
        count = text.count(keyword)

        if count > 0:
            results[path.name] = count

    return results
```


## 4. `client.py` の役割

`client.py` は、Serverに登録されたToolを取得し、LLMへ利用可能なToolとして渡し、LLMが選択したToolを実行する側である。

大きな流れは次の通り。

```text
server.py の mcp を読み込む
↓
MCP Clientを作成
↓
list_tools() でTool一覧を取得
↓
MCP Tool情報をOpenAI Function Calling形式へ変換
↓
質問とTool情報をLLMへ渡す
↓
LLMがTool名と引数を選択
↓
call_tool() でTool実行
↓
Tool結果をLLMへ返す
↓
自然文の最終回答を生成
```


### 4-1. `client.py` で使うライブラリ

```python
import asyncio
import json

from openai import OpenAI
from mcp import Client
from server import mcp
```

それぞれの役割は次の通り。

```text
asyncio
→ .pyファイルから非同期処理を開始するために使用

json
→ LLMから返されたJSON形式のTool引数をPythonのdictへ変換

OpenAI
→ OpenAI APIを呼び出すClient

Client
→ MCP ServerのTool情報取得・Tool実行に使用

from server import mcp
→ server.pyで作成したMCP Serverオブジェクトを読み込む
```

ここでは「Client」という役割が2種類登場するため、区別して考える。

```text
client_openai = OpenAI()
→ OpenAI APIと通信するClient

async with Client(mcp) as client:
→ MCP Serverとやり取りするMCP Client
```

つまり、

```text
OpenAI Client
→ LLMへ質問・Tool情報・Tool結果を渡す

MCP Client
→ MCP ServerからTool情報を取得し、Toolを実行する
```

という役割分担になっている。

その後、

```python
client_openai = OpenAI()
```

としてOpenAI APIを利用するClientを作成する。


### 4-2. `async def ask_mcp(question: str):` とは何か

```python
async def ask_mcp(question: str):
```

は、**質問を受け取り、Tool選択・Tool実行・最終回答生成までをまとめて行う非同期関数**として定義している。

`question: str` の `question` には、後で `main()` から質問文が渡される。

```text
main()で質問文を用意
↓
ask_mcp(質問文)
↓
questionとして受け取る
```

`ask_mcp()` の中では、

```text
1. MCP ServerからTool情報を取得
2. OpenAI形式へ変換
3. LLMへ質問とTool情報を渡す
4. LLMが選んだTool名・引数を取得
5. MCP Toolを実行
6. Tool結果をLLMへ返す
7. 最終回答をreturn
```

という一連の処理を行う。


### 4-3. なぜ `if / elif` の分岐を書かなくてよいのか

通常、自分でToolを選択するなら、例えば次のような分岐を書く方法もある。

```python
if "ファイル一覧" in question:
    # list_filesを使う
elif "回数" in question:
    # count_keywordを使う
```

しかし今回のClientには、このようなToolごとの分岐を書いていない。

代わりに、

```python
response = client_openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": question}
    ],
    tools=openai_tools
)
```

で、**質問と利用可能なTool情報をまとめてLLMへ渡している**。

ここが、実質的にTool選択が行われる箇所である。

LLMはToolの、

```text
name
description
parameters
```

と質問内容を見て、利用するToolと引数を判断する。

その判断結果が、

```python
tool_call = response.choices[0].message.tool_calls[0]
```

へ返される。

その後は、

```python
tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)

tool_result = await client.call_tool(
    tool_name,
    tool_args
)
```

という共通処理でToolを実行する。

つまり、

```text
Client自身がif / elifでToolを選択
```

しているのではなく、

```text
ClientがTool情報をLLMへ渡す
↓
LLMがToolを選択
↓
Clientは選ばれたToolを共通処理で実行
```

という構成になっている。

このため、`count_keyword` のような新しいToolをServerへ追加して `list_tools()` で取得できれば、Client側へそのTool専用の `if / elif` を追加せずに利用できる。


### 4-4. `tool_call` と `tool_result` の違い

この2つは似て見えるが、役割が異なる。

```text
tool_call
→ LLMが返した「このToolを、この引数で呼び出してほしい」という指示

tool_result
→ MCP Clientが実際にToolを実行した結果
```

例えば、

```python
tool_call = response.choices[0].message.tool_calls[0]
```

の時点では、まだToolは実行されていない。

その後、

```python
tool_result = await client.call_tool(
    tool_name,
    tool_args
)
```

を実行して初めて、Server側のToolが動作する。

```text
LLM
↓
tool_call
↓
MCP Client
↓
call_tool()
↓
Tool実行
↓
tool_result
```


### 4-5. OpenAI APIを2回呼び出す理由

今回の `client.py` では、OpenAI APIを主に2回呼び出している。

```text
1回目
→ 質問とTool情報をLLMへ渡す
→ LLMが利用するToolと引数を判断

2回目
→ MCP Toolの実行結果をLLMへ返す
→ Tool結果からユーザー向けの自然文を生成
```

1回目の時点では、LLMはToolを実行していない。

```text
LLM
→ どのToolを使うか判断

MCP Client / Tool
→ 実際の処理を実行

LLM
→ Tool結果を自然文にまとめる
```

という役割分担になっている。


### 4-6. `main()` と `asyncio.run(main())`

`client.py` では、`main()` を処理の入口としている。

```python
async def main():
    answer = await ask_mcp(
        r"C:\python\MCP\data の中から、申請という文字を含む行を探してください。"
    )

    print(answer)
```

ここで、

```python
ask_mcp(
    "質問文"
)
```

へ渡した文字列が、

```python
async def ask_mcp(question: str):
```

の `question` に渡される。

```text
main()
↓
質問文をask_mcp()へ渡す
↓
questionとして受け取る
↓
OpenAI APIへ質問文を渡す
↓
Tool選択・Tool実行
↓
最終回答をreturn
↓
answerへ格納
↓
print(answer)
```

そして、`.py` ファイルの最後に、

```python
asyncio.run(main())
```

を書く。

これは、**非同期関数として定義した `main()` を通常のPythonスクリプトから実際に開始するための入口**である。

```text
python client.py
↓
Pythonがファイルを上から読む
↓
async def main() を定義
↓
asyncio.run(main())
↓
main() が実行される
↓
ask_mcp() が実行される
```

Notebookではトップレベルで `await` を利用できるが、通常の `.py` ファイルでは `asyncio.run(main())` を使って非同期処理を開始する。


### 4-7. 今回の `client.py`

```python
import asyncio
import json

from openai import OpenAI
from mcp import Client
from server import mcp


client_openai = OpenAI()


async def ask_mcp(question: str):
    async with Client(mcp) as client:
        tools = await client.list_tools()

        openai_tools = []

        for tool in tools.tools:
            openai_tools.append(
                {
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description or "",
                        "parameters": tool.input_schema,
                    },
                }
            )

        response = client_openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "user",
                    "content": question
                }
            ],
            tools=openai_tools
        )

        tool_call = response.choices[0].message.tool_calls[0]

        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        tool_result = await client.call_tool(
            tool_name,
            tool_args
        )

        tool_output = tool_result.content[0].text

        final_response = client_openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "user",
                    "content": question
                },
                response.choices[0].message,
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_output
                }
            ]
        )

        return final_response.choices[0].message.content


async def main():
    answer = await ask_mcp(
        r"C:\python\MCP\data の中から、申請という文字を含む行を探してください。"
    )

    print(answer)


asyncio.run(main())
```


## 5. NotebookでMCPの動作を検証

Notebookでは、`client.py` にまとめる前に各処理をセル単位で確認する。

```text
1. Tool一覧を取得
2. Tool情報を確認
3. OpenAI形式へ変換
4. LLMに質問
5. LLMが選んだToolを確認
6. MCP ClientからTool実行
7. Tool結果を確認
8. Tool結果をLLMへ返して最終回答
```

Notebookを使うことで、どの段階で何が行われているのかを確認しやすい。


### 5-1. ライブラリの読み込み


In [1]:
import json

from openai import OpenAI
from mcp import Client
from server import mcp


### 5-2. テスト用文書の作成

複数Toolの使い分けを確認するため、簡単な社内文書を想定したテキストファイルを作成する。


In [2]:
from pathlib import Path

base_path = Path(r"C:\python\MCP\data")
base_path.mkdir(exist_ok=True)

documents = {
    "有給休暇.txt": """有給休暇について
有給休暇は勤怠システムから申請します。
申請後は所属長の承認が必要です。
残日数は勤怠システムから確認できます。
""",

    "PC申請.txt": """PC申請について
新しいPCが必要な場合は情報システム部へ申請します。
申請時には利用目的と必要スペックを記載します。
承認後にPCが手配されます。
""",

    "障害対応.txt": """障害対応について
システム障害が発生した場合は一次切り分けを実施します。
復旧できない場合は担当部署へ連絡します。
重大障害の場合は管理者へ報告します。
""",

    "AI利用ルール.txt": """生成AI利用ルール
生成AIへ機密情報を入力してはいけません。
社外秘文書を生成AIへアップロードしてはいけません。
業務利用する場合は社内ルールを確認してください。
""",

    "セキュリティ.txt": """情報セキュリティについて
パスワードを他人と共有してはいけません。
機密情報の取り扱いには注意してください。
不審なメールを受信した場合は情報システム部へ報告してください。
"""
}

for filename, content in documents.items():
    path = base_path / filename
    path.write_text(content, encoding="utf-8")

print("テスト文書を作成しました。")


テスト文書を作成しました。


### 5-3. Serverが公開しているTool情報を取得

`list_tools()` を使って、MCP Serverに登録されているTool情報を取得する。

ここでは、Tool名だけでなく、docstring由来の `description` と、型ヒントなどから作られた `input_schema` も確認する。


In [3]:
async with Client(mcp) as client:
    tools = await client.list_tools()

for tool in tools.tools:
    print(tool.name)
    print(tool.description)
    print(tool.input_schema)
    print("-" * 50)


read_text_file
指定したテキストファイルの内容を読み込みます。
{'type': 'object', 'properties': {'path': {'title': 'Path', 'type': 'string'}}, 'required': ['path'], 'title': 'read_text_fileArguments'}
--------------------------------------------------
list_files
指定したフォルダ内のファイル一覧を取得します。
{'type': 'object', 'properties': {'path': {'title': 'Path', 'type': 'string'}}, 'required': ['path'], 'title': 'list_filesArguments'}
--------------------------------------------------
get_file_info
指定したファイルの基本情報を取得します。
{'type': 'object', 'properties': {'path': {'title': 'Path', 'type': 'string'}}, 'required': ['path'], 'title': 'get_file_infoArguments'}
--------------------------------------------------
search_folder_text
指定したフォルダ内のテキストファイルから、キーワードを含む行を検索します。
{'type': 'object', 'properties': {'folder_path': {'title': 'Folder Path', 'type': 'string'}, 'keyword': {'title': 'Keyword', 'type': 'string'}}, 'required': ['folder_path', 'keyword'], 'title': 'search_folder_textArguments'}
-----------------------------------------------

### 5-4. MCP Tool情報をOpenAI Function Calling形式へ変換

MCPから取得したTool情報をOpenAI APIへ渡せる形式に変換する。

```text
MCP                         OpenAI Function Calling

tool.name        →          function.name
tool.description →          function.description
tool.input_schema→          function.parameters
```

ここで変換しているのはTool本体ではなく、**LLMへ「利用可能なTool」を説明するためのメタデータ**である。


In [4]:
openai_tools = []

for tool in tools.tools:
    openai_tools.append(
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.input_schema,
            },
        }
    )


### 5-5. LLMにToolを選択させる

質問と利用可能なTool情報をOpenAI APIへ渡す。

この時点では、まだMCP Toolは実行されていない。

LLMは質問内容とTool情報から、

```text
どのToolを使うか
どの引数を渡すか
```

を判断する。


In [5]:
client_openai = OpenAI()

question = r"C:\python\MCP\data の中にはどんなファイルがありますか？"

response = client_openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": question
        }
    ],
    tools=openai_tools
)

tool_call = response.choices[0].message.tool_calls[0]

print("Tool名:", tool_call.function.name)
print("引数:", tool_call.function.arguments)


[09/12/26 14:33:30] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=6836531;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=6836532;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py#1085\1085]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Tool名: list_files
引数: {"path":"C:\\python\\MCP\\data"}


### 5-6. 選択されたToolを実行

`tool_call` からTool名と引数を取り出す。

LLMが返す引数はJSON文字列なので、

```python
json.loads(...)
```

でPythonのdictへ変換してから `call_tool()` へ渡す。


In [6]:
tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)

async with Client(mcp) as client:
    tool_result = await client.call_tool(
        tool_name,
        tool_args
    )

print(tool_result.content)


[TextContent(type='text', text='AI利用ルール.txt', annotations=None, meta=None), TextContent(type='text', text='PC申請.txt', annotations=None, meta=None), TextContent(type='text', text='セキュリティ.txt', annotations=None, meta=None), TextContent(type='text', text='有給休暇.txt', annotations=None, meta=None), TextContent(type='text', text='障害対応.txt', annotations=None, meta=None)]


### 5-7. 複数Toolの使い分け

同じClientへ異なる質問を渡し、LLMが質問内容に応じて異なるToolを選択することを確認する。

```text
ファイル一覧を知りたい
→ list_files

ファイル内容を読みたい
→ read_text_file

複数ファイルから文字列を検索したい
→ search_folder_text
```


In [7]:
async def run_mcp_test(question: str):
    response = client_openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": question
            }
        ],
        tools=openai_tools
    )

    tool_call = response.choices[0].message.tool_calls[0]

    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)

    async with Client(mcp) as client:
        tool_result = await client.call_tool(
            tool_name,
            tool_args
        )

    print("質問:", question)
    print("選択Tool:", tool_name)
    print("引数:", tool_args)
    print("結果:", tool_result.content)


In [8]:
await run_mcp_test(
    r"C:\python\MCP\data の中にはどんなファイルがありますか？"
)

print("-" * 50)

await run_mcp_test(
    r"C:\python\MCP\data\AI利用ルール.txt の内容を確認してください。"
)

print("-" * 50)

await run_mcp_test(
    r"C:\python\MCP\data の中から、「申請」という文字を含む行を探してください。"
)


[09/12/26 14:33:32] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=6836537;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=6836538;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py#1085\1085]8;;\
                             "HTTP/1.1 200 OK"                                                                     

質問: C:\python\MCP\data の中にはどんなファイルがありますか？
選択Tool: list_files
引数: {'path': 'C:\\python\\MCP\\data'}
結果: [TextContent(type='text', text='AI利用ルール.txt', annotations=None, meta=None), TextContent(type='text', text='PC申請.txt', annotations=None, meta=None), TextContent(type='text', text='セキュリティ.txt', annotations=None, meta=None), TextContent(type='text', text='有給休暇.txt', annotations=None, meta=None), TextContent(type='text', text='障害対応.txt', annotations=None, meta=None)]
--------------------------------------------------


[09/12/26 14:33:33] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=6836543;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=6836544;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py#1085\1085]8;;\
                             "HTTP/1.1 200 OK"                                                                     

質問: C:\python\MCP\data\AI利用ルール.txt の内容を確認してください。
選択Tool: read_text_file
引数: {'path': 'C:\\python\\MCP\\data\\AI利用ルール.txt'}
結果: [TextContent(type='text', text='生成AI利用ルール\n生成AIへ機密情報を入力してはいけません。\n社外秘文書を生成AIへアップロードしてはいけません。\n業務利用する場合は社内ルールを確認してください。\n', annotations=None, meta=None)]
--------------------------------------------------


[09/12/26 14:33:34] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=6836549;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=6836550;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py#1085\1085]8;;\
                             "HTTP/1.1 200 OK"                                                                     

質問: C:\python\MCP\data の中から、「申請」という文字を含む行を探してください。
選択Tool: search_folder_text
引数: {'folder_path': 'C:\\python\\MCP\\data', 'keyword': '申請'}
結果: [TextContent(type='text', text='{\n  "PC申請.txt": [\n    "PC申請について",\n    "新しいPCが必要な場合は情報システム部へ申請します。",\n    "申請時には利用目的と必要スペックを記載します。"\n  ],\n  "有給休暇.txt": [\n    "有給休暇は勤怠システムから申請します。",\n    "申請後は所属長の承認が必要です。"\n  ]\n}', annotations=None, meta=None)]


### 5-8. Tool追加による拡張性を確認

`server.py` にPython関数 `count_keyword` を追加し、`@mcp.tool()` でTool登録する。

その後、

```text
Kernel再起動
↓
list_tools() でTool情報を再取得
↓
OpenAI形式へ変換
↓
質問とTool一覧をLLMへ渡す
↓
LLMが count_keyword を選択
```

という流れを確認する。

Client側では `count_keyword` 専用の `if / elif` は追加していない。


In [9]:
async with Client(mcp) as client:
    tools = await client.list_tools()

openai_tools = []

for tool in tools.tools:
    openai_tools.append(
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.input_schema,
            },
        }
    )

question = r"C:\python\MCP\data の中で、「申請」という文字が各ファイルに何回出てくるか調べてください。"

response = client_openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": question}
    ],
    tools=openai_tools
)

tool_call = response.choices[0].message.tool_calls[0]

print("Tool名:", tool_call.function.name)
print("引数:", tool_call.function.arguments)


[09/12/26 14:33:36] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=6836555;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=6836556;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py#1085\1085]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Tool名: count_keyword
引数: {"folder_path":"C:\\python\\MCP\\data","keyword":"申請"}


### 5-9. Tool実行結果をLLMへ返す

LLMが選択したToolを実行し、その結果を2回目のOpenAI API呼び出しでLLMへ返す。

これにより、Toolの生の結果ではなく、ユーザー向けの自然文として回答できる。


In [10]:
tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)

async with Client(mcp) as client:
    tool_result = await client.call_tool(
        tool_name,
        tool_args
    )

tool_output = tool_result.content[0].text

final_response = client_openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": question
        },
        response.choices[0].message,
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_output
        }
    ]
)

print(final_response.choices[0].message.content)


[09/12/26 14:33:37] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=6836561;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=6836562;file://C:\ProgramData\anaconda3\envs\mcp\Lib\site-packages\httpx2\_client.py#1085\1085]8;;\
                             "HTTP/1.1 200 OK"                                                                     

フォルダ「C:\python\MCP\data」の中で、「申請」という文字は以下のファイルに以下の回数出てきます。

- PC申請.txt: 3回
- 有給休暇.txt: 2回

他のファイルには「申請」という文字は含まれていませんでした。


## 6. `client.py` をターミナルから実行する

Notebookで一連の流れを確認した後、同じ処理を `client.py` にまとめて単独実行する。

今回の環境ではAnaconda Promptを使用したが、PowerShellやコマンドプロンプトなど、対象のPython環境を有効化できるターミナルでも実行できる。

ターミナルから、

```text
python client.py
```

と実行する。

これはMCP専用の操作ではなく、通常のPythonスクリプトをPythonから実行しているだけである。


### 6-1. ターミナル側にもPython環境は必要

MCPがターミナルへPython環境を自動で用意するわけではない。

今回の場合はAnaconda Promptで `mcp` 環境を有効化してから実行する。

```text
Anaconda Prompt
↓
conda環境 mcp を有効化
↓
C:\python\MCP へ移動
↓
python client.py
```

`mcp` 環境には、Python・MCP・OpenAIなど、`client.py` の実行に必要なライブラリが入っている必要がある。


### 6-2. `python client.py` を実行すると何が起きるか

```text
python client.py
↓
Pythonがclient.pyを上から読み込む
↓
from server import mcp
↓
server.pyがPythonモジュールとして読み込まれる
↓
mcp = MCPServer(...) が実行される
↓
@mcp.tool() の付いた関数がToolとして登録される
↓
MCP Clientを作成
↓
list_tools() でTool情報取得
↓
OpenAIへ質問とTool情報を渡す
↓
LLMがToolを選択
↓
call_tool() でToolを実行
↓
Tool結果をOpenAIへ返す
↓
最終回答をターミナルへ表示
```

今回の構成では `client.py` が、

```python
from server import mcp
```

によって `server.py` を読み込む。

Pythonではモジュールをimportすると、そのモジュールのトップレベルのコードが読み込まれる。

そのため `server.py` 内の、

```python
mcp = MCPServer(...)
```

によってServerオブジェクトが作成され、`@mcp.tool()` の付いた関数がToolとして登録された状態の `mcp` を `client.py` から利用できる。

つまりMCPが別のクラウド環境へ自動接続しているのではなく、**用意済みのPython環境で `client.py` が `server.py` を読み込み、そのServerオブジェクトをMCP Clientから利用している**。


### 6-3. なぜNotebookで確認した後にターミナルでも実行するのか

Notebookと `client.py` では目的が異なる。

```text
Notebook
→ 処理を分解し、各段階を理解・確認する

client.py
→ 一連の処理を1本のPythonプログラムとして実行する
```

NotebookだけでもMCP処理は確認できる。

最後に `client.py` をターミナルから実行することで、

**Notebookに依存せず、1つのPython ClientとしてTool選択から最終回答まで動作する**

ことを確認している。


### 6-4. Serverを書き換えたときの注意

Notebookでは、一度importした `server.py` の内容がKernel内に残る。

そのため、`server.py` にToolを追加・変更した場合は、変更がNotebookへ反映されないことがある。

その場合は、

```text
server.pyを保存
↓
NotebookのKernelを再起動
↓
必要なセルを最初から再実行
```

する。

一方、ターミナルから新しく、

```text
python client.py
```

を実行した場合は新しいPythonプロセスが開始されるため、その時点の `server.py` と `client.py` が読み込まれる。


## 7. 今回の実装で分かったMCPの便利さ

今回の実装で特に分かりやすかった利点は、**Server側で公開するToolを共通形式でClientから取得・実行できること**である。

今回のClientは、

```python
tools = await client.list_tools()
```

でServerが公開するTool情報を取得する。

その後、取得した、

```text
name
description
input_schema
```

をOpenAI Function Calling形式へ変換し、LLMへ渡している。

ここで役割を分けて考えると分かりやすい。

```text
MCP
→ Toolを共通形式で公開・取得・実行する仕組み

LLM
→ 質問内容とTool情報から、どのToolを使うか判断する
```

つまり、**MCP自身が質問文を読んでToolを選んでいるわけではない。**

今回の実装ではOpenAIのLLMがToolを選択し、MCP Clientが選ばれたToolを実行している。

そのため、

```text
この質問なら list_files
この質問なら search_folder_text
この質問なら count_keyword
```

というToolごとの分岐をClientへ固定的に書く必要がない。

`count_keyword` を追加したときも、

```text
Server側にPython関数を追加
↓
@mcp.tool() でTool登録
↓
Clientがlist_tools()で取得
↓
LLMへTool情報を渡す
↓
LLMが必要に応じて新Toolを選択
↓
Client.call_tool()で実行
```

という同じ流れで利用できた。

つまり今回確認できたMCPの利点は、

**異なる処理をToolという共通形式で公開し、Client側から統一的に発見・実行しやすくすること**

である。


## 8. 今回の実装範囲と今後の改善点

今回のコードは、MCPの基本的なTool連携を理解することを優先したシンプルな実装である。

そのため、実運用を想定すると追加で考慮すべき点がある。

```text
Tool呼び出し
→ 現在は tool_calls[0] を使用し、1回に1つのTool呼び出しを前提としている

Toolを使わない回答
→ LLMがToolを選択しなかった場合の分岐は未実装

エラー処理
→ Tool実行失敗、ファイル不存在、API失敗などの詳細な例外処理は簡略化

Tool結果
→ 今回は tool_result.content[0].text を主に利用
→ 実運用では structured_content、is_error、複数contentなども考慮できる

質問入力
→ client.pyでは動作確認用の質問文を固定
→ 実アプリではCLI、Web UI、チャットUIなどから受け取る形に発展可能

接続方式
→ 今回は Client(mcp) による同一プロセス接続
→ 用途に応じてstdioやStreamable HTTPによる接続へ発展可能
```

今回はこれらをあえて複雑化せず、まず、

```text
ServerでTool公開
↓
ClientでTool情報取得
↓
LLMでTool選択
↓
MCP ClientでTool実行
↓
LLMで最終回答
```

という基本フローを確認することを目的とした。


## 9. 今回確認できたこと

今回の実装では、次の一連の流れを確認した。

```text
Python関数を作成
↓
@mcp.tool() でMCP Toolとして登録
↓
Clientがlist_tools()でTool情報を取得
↓
OpenAI Function Calling形式へ変換
↓
質問とTool情報をLLMへ渡す
↓
LLMがToolと引数を選択
↓
Client.call_tool()でTool実行
↓
Tool結果をLLMへ返す
↓
自然文の最終回答を生成
```

さらに、Serverへ `count_keyword` を追加した際にも、Client側へ専用の分岐処理を追加せず利用できることを確認した。

今回の構成では、MCP Client・MCP Server・Tool・テストデータを同一PC・同一Pythonプロセス上で動かしている。

一方、MCP Serverは同一PCに限定されるものではなく、stdioを利用したローカル別プロセスや、Streamable HTTPを利用したクラウド・社内サーバー・オンプレ環境などへ発展させることもできる。

MCPはクラウドAPIそのものではなく、**Clientと機能提供側を共通の形式で接続するための仕組み**である。

今回の実装では、

```text
MCP
→ Toolの公開・発見・実行を共通化

OpenAI LLM
→ 質問内容に応じたTool選択

Python
→ Tool本体の処理
```

という役割分担も確認できた。

Notebookでは処理を段階的に検証し、最後に `client.py` としてまとめてターミナルから実行することで、単独のPython Clientとして動作することまで確認した。
